# Notebook 07: Finite Element Baseline Analysis (1.0 kN Normalized Load)
**Project**: Stegoceras Biomechanics & Uncertainty Quantification  
**Specimen**: *Stegoceras validum* (UALVP 2, referred specimen)  
**Deliverable**: Phase 4 Primary Benchmark Solve  

### Objective
Execute the primary $1.0\text{ kN}$ broad compressive load ($3000\text{ mm}^2$ patch) benchmark solve under homogeneous linear elasticity ($E = 17.0\text{ GPa}, \nu = 0.30$), verify exact force and moment equilibrium, and extract quantitative stress, strain, and deformation metrics across 6 anatomical subregions.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from stegoceras_biomechanics.fea.meshing import extract_boundary_surface
from stegoceras_biomechanics.fea.loads import generate_dome_load_patch
from stegoceras_biomechanics.fea.boundary_conditions import generate_boundary_constraints
from stegoceras_biomechanics.fea.solver import solve_linear_elasticity
from stegoceras_biomechanics.fea.validation import verify_global_equilibrium
from stegoceras_biomechanics.fea.results import extract_subregion_metrics

med_data = np.load('../data/meshes/cleaned/stegoceras_tetmesh_medium.npz')
nodes = med_data['nodes']
elements = med_data['elements']
surf = extract_boundary_surface(nodes, elements)

loaded_nodes, nodal_forces, _, load_spec = generate_dome_load_patch(surf, 3000.0, 1000.0)
condyle_nodes, nuchal_nodes, _ = generate_boundary_constraints(surf)

print('Solving 3D linear elasticity on medium mesh...')
sol = solve_linear_elasticity(
    nodes=nodes,
    elements=elements,
    youngs_modulus_MPa=17000.0,
    poisson_ratio=0.30,
    loaded_node_indices=loaded_nodes,
    nodal_forces_N=nodal_forces,
    condyle_node_indices=condyle_nodes,
    nuchal_node_indices=nuchal_nodes,
    solver_method='direct',
)

eq_check = verify_global_equilibrium(sol, load_spec)
max_disp = float(np.max(sol.displacement_magnitudes_mm))
max_vm = float(np.max(sol.nodal_von_mises_MPa))
p95_vm = float(np.percentile(sol.nodal_von_mises_MPa, 95))

print('=== Global Equilibrium & Performance ===')
print(f'Max Cranial Displacement: {max_disp*1000:.2f} μm')
print(f'Max von Mises Stress: {max_vm:.2f} MPa')
print(f'95th Percentile Stress: {p95_vm:.2f} MPa')
print(f'Total Strain Energy: {sol.total_strain_energy_mJ:.4f} mJ')
print(f'Force Residual: {eq_check.residual_force_norm_N:.6f} N ({eq_check.residual_force_relative_pct:.6f}%)')
print(f'Moment Residual: {eq_check.residual_moment_norm_Nmm:.4f} N*mm')
assert eq_check.is_force_balanced, 'Force equilibrium failed!'
assert eq_check.is_moment_balanced, 'Moment equilibrium failed!'
print('✓ Exact static equilibrium verified!')

Transforming over 1000 vertices to C_CONTIGUOUS.


Transforming over 1000 elements to C_CONTIGUOUS.


Solving 3D linear elasticity on medium mesh...


=== Global Equilibrium & Performance ===
Max Cranial Displacement: 25.49 μm
Max von Mises Stress: 11.01 MPa
95th Percentile Stress: 1.26 MPa
Total Strain Energy: 5.3818 mJ
Force Residual: 0.000000 N (0.000000%)
Moment Residual: 0.0000 N*mm
✓ Exact static equilibrium verified!


### Anatomical Subregion Stress & Strain Metrics
Inspect the regional mechanical distribution across the 6 anatomical skull subregions.

In [2]:
df_metrics = pd.read_csv('../results/phase4/ualvp2_1kn_subregion_metrics.csv')
display_cols = ['region_name', 'num_nodes', 'max_von_mises_MPa', 'p95_von_mises_MPa', 'mean_von_mises_MPa', 'max_displacement_mm', 'regional_strain_energy_mJ']
df_metrics[display_cols]

,region_name,num_nodes,max_von_mises_MPa,p95_von_mises_MPa,mean_von_mises_MPa,max_displacement_mm,regional_strain_energy_mJ
0,Whole Skull (Global),189696,11.010404,1.262267,0.405582,0.025495,5.381832
1,Frontoparietal Dome Apex,10078,5.831319,1.002062,0.319396,0.023244,0.049026
2,Sub-Dome Vault Core,58331,7.498046,1.577172,0.649186,0.020473,1.366192
3,Endocranial Braincase Roof,9337,3.429033,1.471679,0.675283,0.011943,0.639871
4,Lateral Cranium,42522,7.498046,1.435040,0.454045,0.023244,0.870881
5,Posterior Skull & Nuchal Shelf,17254,4.129470,1.351757,0.403249,0.001636,0.650382
6,Basicranium & Condyle,14420,11.010404,0.636438,0.303157,0.012050,1.867936
